# Transformers From Scratch — Part 1
**Data Pipeline + Bigram Model + The Moment Attention Becomes Inevitable**

---

## The Story Behind This Notebook

```
  I needed to build two specialized AI models for production.
  I knew I needed LoRA for fine-tuning.
  I knew I needed prefix caching to warm up the KV cache.
  I knew reinforcement learning with human feedback was somewhere in the picture.

  But when I sat down to actually build — I realized I couldn't answer
  the simplest questions:

  Why LoRA and not full fine-tuning?
  Why does the cache need warming up at all?
  What is actually happening inside the model when I call model.generate()?

  So I went back to the drawing board. Notebooks. From scratch.
  This is Part 1.
```

## What this notebook covers

```
  PART A — Data Pipeline:
  Text → characters → integers → tensors → GPU → training batches

  PART B — Bigram Model:
  Embeddings → logits → cross-entropy loss → training loop

  THE MOMENT:
  Why P(next | 'h','e','l') = P(next | 'l') in this model
  and why that single equation motivates the entire transformer.
```

---

## Full Data Flow

```
  "hello world"
       │
       ▼  set() + sorted()
  [' ', 'd', 'e', 'h', 'l', 'o', 'r', 'w']   ← vocabulary
       │
       ▼  stoi dict
  [3, 2, 4, 4, 5, 0, 7, 5, 6, 4, 1]          ← integer token IDs
       │
       ▼  torch.tensor(dtype=torch.long)
  tensor([3, 2, 4, 4, 5, ...])                ← 1D tensor on CPU
       │
       ▼  .to(device)
  tensor([...]) on cuda:0                      ← GPU memory
       │
       ▼  get_batch()
  x: (B, T) inputs    y: (B, T) targets        ← training pairs
       │
       ▼  nn.Embedding
  logits: (B, T, vocab_size)                   ← raw scores
       │
       ▼  F.cross_entropy
  loss: scalar                                 ← how wrong we are
       │
       ▼  .backward() + optimizer.step()
  updated weights                              ← model learns
```

---
# PART A — Data Pipeline
## Step 1 — Text → Unique Characters

In [1]:
# set(text): unordered collection of unique characters
# 'l' appears 3 times in 'hello' → set keeps only 1
# sorted(): makes order deterministic (by ASCII value) — same text = same vocab every run

text  = "hello world. hello transformer."
chars = sorted(set(text))          # → [' ', '.', 'a', 'd', 'e', 'f', 'h', ...]
vocab_size = len(chars)            # → 15

print(f"Text       : '{text}'")
print(f"Vocab size : {vocab_size}")
print(f"Characters : {chars}")

Text       : 'hello world. hello transformer.'
Vocab size : 15
Characters : [' ', '.', 'a', 'd', 'e', 'f', 'h', 'l', 'm', 'n', 'o', 'r', 's', 't', 'w']


## Step 2 — Build Lookup Dictionaries

In [2]:
# stoi: string → integer  (char to index)
# itos: integer → string  (index to char)
# encode / decode: convenience wrappers

stoi  = {ch: i for i, ch in enumerate(chars)}   # → {' ':0, '.':1, 'a':2, ...}
itos  = {i: ch for i, ch in enumerate(chars)}   # → {0:' ', 1:'.', 2:'a', ...}
encode = lambda s: [stoi[c] for c in s]          # text  → list of ints
decode = lambda l: ''.join([itos[i] for i in l]) # ints  → text

print(f"encode('hello') = {encode('hello')}")
print(f"decode back     = '{decode(encode('hello'))}'")
print()
print("Vocabulary mapping:")
for ch, idx in stoi.items():
    print(f"  '{ch}' ↔ {idx}")

encode('hello') = [6, 4, 7, 7, 10]
decode back     = 'hello'

Vocabulary mapping:
  ' ' ↔ 0
  '.' ↔ 1
  'a' ↔ 2
  'd' ↔ 3
  'e' ↔ 4
  'f' ↔ 5
  'h' ↔ 6
  'l' ↔ 7
  'm' ↔ 8
  'n' ↔ 9
  'o' ↔ 10
  'r' ↔ 11
  's' ↔ 12
  't' ↔ 13
  'w' ↔ 14


## Step 3 — Text → Tensor

```
  WHY dtype=torch.long?
  nn.Embedding uses integers as ROW INDICES into a weight matrix.
  Floats cannot be row indices — PyTorch requires int64 (torch.long).

  WHY .to(device)?
  Data and model MUST be on the same device.
  CPU tensor + GPU model → runtime error.
  .to(device) returns a NEW tensor on the target device.
  Original tensor unchanged.
```

In [3]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# encode text → integer list → tensor
data = torch.tensor(encode(text), dtype=torch.long)  # shape (31,)
data = data.to(device)                                # move to GPU

print(f"\ndata shape  : {data.shape}")
print(f"data dtype  : {data.dtype}")
print(f"data device : {data.device}")
print(f"first 5     : {data[:5].tolist()} = '{decode(data[:5].tolist())}'")  

Device: cuda
GPU   : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM  : 8.6 GB

data shape  : torch.Size([31])
data dtype  : torch.int64
data device : cuda:0
first 5     : [6, 4, 7, 7, 10] = 'hello'


## Step 4 — Build Training Batches

```
  WHY mini-batches and not the whole dataset?
  - Real datasets = billions of tokens — won't fit in memory
  - Noisy gradient from one random batch is better than waiting
    for one perfect pass over all data (mini-batch SGD)
  - The noise actually helps: escapes local minima, acts as regularization

  block_size = 4   (tokens per training example)
  batch_size = 3   (sequences per gradient step)
  predictions per step = block_size × batch_size = 12

  x (inputs):          y (targets = x shifted +1):
  ┌──────────────┐     ┌──────────────┐
  │ h  e  l  l  │     │ e  l  l  o  │  ← predict next char at each position
  │ ' ' w  o  r │     │ w  o  r  l  │
  │ l  l  o  ' '│     │ l  o  ' ' w │
  └──────────────┘     └──────────────┘
```

In [4]:
block_size = 4
batch_size = 3

def get_batch(data, block_size, batch_size, device='cpu'):
    # randint(len-block_size): subtract to avoid running off the end of data
    ix = torch.randint(len(data) - block_size, (batch_size,))  # → shape (3,)

    x = torch.stack([data[i  :i+block_size  ] for i in ix])    # → shape (3, 4)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])    # → shape (3, 4)

    return x.to(device), y.to(device)

torch.manual_seed(0)
xb, yb = get_batch(data, block_size, batch_size, device)

print(f"x shape: {xb.shape}   (B={batch_size}, T={block_size})")
print(f"y shape: {yb.shape}")
print()
print("Every (context → next token) pair in this batch:")
for b in range(batch_size):
    for t in range(block_size):
        ctx = decode([xb[b,i].item() for i in range(t+1)])
        tgt = decode([yb[b,t].item()])
        print(f"  '{ctx}' → '{tgt}'")
    print()

x shape: torch.Size([3, 4])   (B=3, T=4)
y shape: torch.Size([3, 4])

Every (context → next token) pair in this batch:
  'r' → 'm'
  'rm' → 'e'
  'rme' → 'r'
  'rmer' → '.'

  'l' → 'd'
  'ld' → '.'
  'ld.' → ' '
  'ld. ' → 'h'

  'r' → 'a'
  'ra' → 'n'
  'ran' → 's'
  'rans' → 'f'



---
# PART B — Bigram Model
## What is an Embedding?

```
  PROBLEM: neural networks do matrix multiplications.
  You cannot multiply a matrix by an integer like 6.
  The integer carries no geometric meaning.

  SOLUTION: replace each integer with a vector of floats.

  BEFORE (integer — just an address):
  token 'h' = 6    (just a number, no meaning)

  AFTER (embedding — a point in N-dimensional space):
  token 'h' = [ 0.12, -0.34,  0.87,  0.03, -0.55, ... ]

  nn.Embedding is a matrix of shape (vocab_size, embedding_dim).
  Each ROW is one token's vector.
  The operation is: return weight[token_id]  — row lookup, nothing more.

  Shape rule:  idx shape (B, T)  →  output shape (B, T, embedding_dim)

  Rows start RANDOM. Training moves them via backprop.
  The loss tells each row: score the correct next token higher.
```

## What are Logits?

```
  Logits = the RAW, UNCONSTRAINED output of the model before softmax.
  Any real number — negative, positive, large, small.
  They are NOT probabilities.

  logits → softmax → probabilities
  softmax(z)_i = exp(z_i) / Σ exp(z_j)

  F.cross_entropy takes logits directly and applies softmax internally.
  Model always outputs logits. Downstream converts as needed.

  In BigramModel:  embedding_dim = vocab_size
  → each embedding row IS directly the logits
  → no other layers needed
  → this is the simplest possible language model
```

In [5]:
import torch.nn as nn
import torch.nn.functional as F
import math

class BigramModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # ONE learnable tensor: shape (vocab_size, vocab_size) = (15, 15)
        # weight[token_id] = 15 logit scores for the next token
        # 225 parameters total — the entire model
        self.embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx: (B, T) integer tensor
        # embedding_table(idx) = weight[idx] — row lookup for every integer
        # (B, T) indices → (B, T, vocab_size) float vectors
        logits = self.embedding_table(idx)   # → (B, T, vocab_size)

        loss = None
        if targets is not None:
            B, T, C = logits.shape           # B=3, T=4, C=15
            # F.cross_entropy needs (N, C) predictions and (N,) targets
            # collapse (3,4,15) → (12,15) and (3,4) → (12,)
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
            # F.cross_entropy: softmax + -log(p_correct) + mean — one fused op

        return logits, loss


torch.manual_seed(42)
model = BigramModel(vocab_size).to(device)

print(f"Parameters    : {sum(p.numel() for p in model.parameters())}")
print(f"Weight shape  : {model.embedding_table.weight.shape}")
print(f"Random baseline loss: {math.log(vocab_size):.4f} nats")

Parameters    : 225
Weight shape  : torch.Size([15, 15])
Random baseline loss: 2.7081 nats


## Forward Pass — Shapes at Every Step

In [6]:
torch.manual_seed(0)
xb, yb = get_batch(data, block_size, batch_size, device)

logits, loss = model(xb, yb)

print(f"Input  xb     : {xb.shape}              ← (B, T) integer token IDs")
print(f"Output logits : {logits.shape}   ← (B, T, vocab_size) raw scores")
print(f"Loss          : {loss.item():.4f}            ← scalar, mean of {batch_size*block_size} predictions")
print(f"Random baseline: {math.log(vocab_size):.4f}")
print(f"Model is currently {'WORSE' if loss.item() > math.log(vocab_size) else 'BETTER'} than random")
print()
print("Loss interpretation:")
print(f"  -log(1/{vocab_size}) = {math.log(vocab_size):.4f}  ← untrained model (uniform distribution)")
print(f"  -log(0.5)  = {-math.log(0.5):.4f}  ← model assigns 50% to correct token")
print(f"  -log(0.9)  = {-math.log(0.9):.4f}  ← model assigns 90% to correct token")
print(f"  -log(1.0)  = {-math.log(1.0):.4f}  ← perfect prediction")

Input  xb     : torch.Size([3, 4])              ← (B, T) integer token IDs
Output logits : torch.Size([3, 4, 15])   ← (B, T, vocab_size) raw scores
Loss          : 3.1634            ← scalar, mean of 12 predictions
Random baseline: 2.7081
Model is currently WORSE than random

Loss interpretation:
  -log(1/15) = 2.7081  ← untrained model (uniform distribution)
  -log(0.5)  = 0.6931  ← model assigns 50% to correct token
  -log(0.9)  = 0.1054  ← model assigns 90% to correct token
  -log(1.0)  = -0.0000  ← perfect prediction


## Training Loop

```
  One step:
  get_batch → forward → zero_grad → backward → step

  AdamW update rule:  weight -= lr × m / √v
    m = running gradient mean    (direction)
    v = running gradient² mean   (magnitude history)

  Large gradient → large v → SMALLER step  (sensitive weight, avoid overshoot)
  Small gradient → small v → LARGER step   (stuck weight, needs nudge)
```

In [7]:
torch.manual_seed(42)
model     = BigramModel(vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

losses = []
for step in range(1000):
    xb, yb      = get_batch(data, block_size, batch_size, device)
    logits, loss = model(xb, yb)      # forward pass
    optimizer.zero_grad()              # clear old gradients — MUST do every step
    loss.backward()                    # backprop: fill .grad on all parameters
    optimizer.step()                   # update: weight -= lr × m/√v
    losses.append(loss.item())

print(f"{'Step':>6}  {'Loss':>8}  {'vs random':>12}")
print("-" * 32)
for s in [0, 100, 300, 500, 799, 999]:
    diff = math.log(vocab_size) - losses[s]
    print(f"{s:>6}  {losses[s]:>8.4f}  {diff:>+12.4f}")
print()
print(f"Random baseline : {math.log(vocab_size):.4f}")
print(f"Final loss      : {losses[-1]:.4f}")
print(f"Improvement     : {math.log(vocab_size)-losses[-1]:.4f} nats")

  Step      Loss     vs random
--------------------------------
     0    3.2569       -0.5489
   100    2.2629       +0.4451
   300    0.9095       +1.7986
   500    0.7560       +1.9521
   799    0.6133       +2.0947
   999    0.7050       +2.0030

Random baseline : 2.7081
Final loss      : 0.7050
Improvement     : 2.0030 nats


---
## Generation — Using the Trained Model

In [8]:
def generate(model, start_tokens, max_new_tokens, device):
    """
    Autoregressively generate tokens one at a time.
    start_tokens: list of integer token IDs
    """
    model.eval()
    context = torch.tensor(start_tokens, dtype=torch.long).unsqueeze(0).to(device)
    # unsqueeze(0): add batch dim → shape (1, T)

    for _ in range(max_new_tokens):
        with torch.no_grad():                       # no gradient tracking during generation
            all_logits, _ = model(context)          # → (1, T, vocab_size)
            logits = all_logits[0, -1, :]           # → (vocab_size,) last position only
            # WHY last position? It has seen all previous tokens as context.
            # Its logits = prediction for the NEXT unknown token.

        probs      = F.softmax(logits, dim=-1)      # logits → probabilities
        next_token = torch.multinomial(probs, 1)    # sample from distribution
        context    = torch.cat([context, next_token.unsqueeze(0)], dim=1)  # append

    return decode(context[0].tolist())


start = encode('h')   # start with 'h'
output = generate(model, start, max_new_tokens=50, device=device)
print(f"Generated: '{output}'")
print()
print("Note: this model only looks at ONE token at a time.")
print("It has no memory of what came before the current position.")

Generated: 'helo . wo trmerllorlllld. heransformelo hero wo hel'

Note: this model only looks at ONE token at a time.
It has no memory of what came before the current position.


---
## THE MOMENT — Why This Model Fails

```
  Run the cell below. It passes three different contexts to the model:
    ['h', 'e', 'l']   — three tokens of context
    ['o', 'r', 'l']   — different first two tokens, same last token
    ['l']             — just the last token alone

  The prediction at the last position should be different.
  'h','e' before 'l' strongly suggests 'l' or 'o' (spelling hello).
  But watch what actually happens.
```

In [ ]:
model.eval()

context_labels = ["'h','e','l'", "'o','r','l'", "'l' alone"]
contexts_to_test = [encode('hel'), encode('orl'), encode('l')]

print("Probability distribution over next token:")
print(f"{'token':<8}", end='')
for label in context_labels:
    print(f"  {label:>12}", end='')
print()
print("-" * 55)

all_probs = []
for ctx in contexts_to_test:
    tensor_ctx = torch.tensor(ctx, dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        logits, _ = model(tensor_ctx)
        probs = F.softmax(logits[0, -1, :], dim=-1)
    all_probs.append(probs.cpu())

for i, ch in enumerate(chars):
    print(f"  '{ch}'   ", end='')
    for probs in all_probs:
        print(f"  {probs[i].item():>12.4f}", end='')
    print()

print()
print("=" * 55)
print("OBSERVATION: all three columns are IDENTICAL.")
print()
print("P(next | 'h','e','l')  =  P(next | 'o','r','l')  =  P(next | 'l')")
print()
print("The model ignores context. Only the LAST token matters.")
print("This is the fundamental failure of the bigram model.")
print("And it is exactly the problem attention was invented to solve.")

---
## Summary

```
  WHAT WE BUILT:
  ┌──────────────────────────────────────────────────────────────┐
  │  Text → integers → tensors → GPU → batches         (Part A) │
  │  Embeddings → logits → loss → training loop        (Part B) │
  │  The fundamental limitation of context-free models          │
  └──────────────────────────────────────────────────────────────┘

  THE KEY INSIGHT:
  P(next | h, e, l)  =  P(next | l)

  This equation is why attention exists.
  This equation is why transformers replaced everything before them.
  This equation is what I had been using in production
  without understanding for two years.

  WHAT COMES NEXT (Part 2):
  Attention — making P(next | h,e,l) genuinely different from P(next | l)
  by letting every token look back at every previous token.
```

| Concept | What it is |
|---------|------------|
| `set()` + `sorted()` | Deterministic vocabulary from raw text |
| `torch.long` | Required dtype for embedding row indices |
| `.to(device)` | Move tensor CPU → GPU, returns new tensor |
| `get_batch()` | Random mini-batch sampler — the engine of SGD |
| `nn.Embedding` | Learnable lookup table: int → float vector |
| Logits | Raw model output before softmax — any real number |
| `F.cross_entropy` | Softmax + `-log(p_correct)` + mean, fused |
| AdamW | Adaptive optimizer — big gradient = smaller step |
| BigramModel limit | `P(next\|context) = P(next\|last token)` — no memory |

**Next → Part 2: Attention** — the fix for everything this model cannot do